# Mission Control AI — Sprint 2 Prompt e IA

Sistema inteligente de monitoramento de uma missão espacial experimental.

O projeto simula dados operacionais da missão, identifica alertas críticos e utiliza IA generativa para gerar uma análise automatizada do status da missão.

In [29]:
# Instala dependência necessária para o Ollama
!sudo apt-get update
!sudo apt-get install -y zstd

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,344 kB]
Fetched 4,476 kB in 1s (3,301 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency t

In [30]:
# Instalação do Ollama no Google Colab
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [31]:
import subprocess
import time

# Inicia o servidor Ollama em segundo plano
processo_ollama = subprocess.Popen(["ollama", "serve"])

# Aguarda alguns segundos para o servidor iniciar
time.sleep(5)

print("Servidor Ollama iniciado")

Servidor Ollama iniciado


In [32]:
# Baixa o modelo Llama 3.2 1B
!ollama pull llama3.2:1b

In [33]:
# Instala a biblioteca Python para usar o Ollama
!pip install ollama -q

import ollama

print("Biblioteca Ollama instalada e importada")

Biblioteca Ollama instalada e importada


In [34]:
import random
import time
import ollama

## Geração dos dados simulados

Nesta etapa, o sistema cria dados simulados para representar o estado atual de uma missão espacial experimental.

Foram usados valores aleatórios para temperatura, energia, comunicação e status dos módulos porque o objetivo desta Sprint é validar a lógica do sistema antes de usar sensores reais. Assim, é possível testar diferentes cenários, desde uma missão estável até uma situação crítica.


In [35]:
def gerar_dados_missao():
    # Os dados são simulados para representar diferentes condições da missão
    dados = {
        "temperatura": random.randint(10, 110),   # em °C
        "energia": random.randint(5, 100),        # em %
        "comunicacao": random.choice(["estável", "instável", "perdida"]),

        "modulos": {
            "suporte_vida": random.choice(["normal", "atenção", "falha"]),
            "navegacao": random.choice(["normal", "atenção", "falha"]),
            "propulsao": random.choice(["normal", "atenção", "falha"])
        }
    }

    return dados

## Análise dos dados e geração de alertas

Depois que os dados são gerados, o sistema compara cada valor com limites definidos no código.

Quando algum parâmetro fica fora da faixa considerada segura, o sistema registra um alerta e sugere uma decisão automática. Essa parte representa a lógica de tomada de decisão básica exigida no desafio.

In [36]:
def analisar_missao(dados):
    alertas = []
    decisoes = []
    nivel_risco = "BAIXO"

    # Análise da temperatura
    if dados["temperatura"] >= 90:
        alertas.append("Temperatura crítica detectada.")
        decisoes.append("Ativar sistema de resfriamento de emergência.")
        nivel_risco = "ALTO"

    elif dados["temperatura"] >= 70:
        alertas.append("Temperatura acima do ideal.")
        decisoes.append("Aumentar ventilação dos módulos.")
        nivel_risco = "MÉDIO"

    # Análise da energia
    if dados["energia"] <= 20:
        alertas.append("Nível de energia crítico.")
        decisoes.append("Ativar modo de economia de energia.")
        nivel_risco = "ALTO"

    elif dados["energia"] <= 40:
        alertas.append("Energia abaixo do ideal.")
        decisoes.append("Reduzir consumo dos sistemas secundários.")

        if nivel_risco != "ALTO":
            nivel_risco = "MÉDIO"

    # Análise da comunicação
    if dados["comunicacao"] == "perdida":
        alertas.append("Comunicação perdida com a base.")
        decisoes.append("Ativar protocolo de reconexão automática.")
        nivel_risco = "ALTO"

    elif dados["comunicacao"] == "instável":
        alertas.append("Comunicação instável.")
        decisoes.append("Redirecionar sinal para antena reserva.")

        if nivel_risco != "ALTO":
            nivel_risco = "MÉDIO"

    # Análise dos módulos da missão
    nomes_modulos = {
        "suporte_vida": "módulo de suporte à vida",
        "navegacao": "módulo de navegação",
        "propulsao": "módulo de propulsão"
    }

    for modulo, status in dados["modulos"].items():
        nome_modulo = nomes_modulos.get(modulo, modulo)

        if status == "falha":
            alertas.append(f"Falha detectada no {nome_modulo}.")
            decisoes.append(f"Isolar o {nome_modulo} e iniciar diagnóstico técnico.")
            nivel_risco = "ALTO"

        elif status == "atenção":
            alertas.append(f"{nome_modulo.capitalize()} requer atenção.")
            decisoes.append(f"Monitorar o {nome_modulo} continuamente.")

            if nivel_risco != "ALTO":
                nivel_risco = "MÉDIO"

    # Caso nenhum problema seja encontrado
    if not alertas:
        alertas.append("Nenhum alerta crítico detectado.")
        decisoes.append("Manter operação padrão da missão.")

    return alertas, decisoes, nivel_risco

## Integração com IA generativa

Depois que o código identifica os alertas e define o nível de risco, a IA é usada para gerar uma explicação mais fácil de entender.

A ideia não é deixar a IA decidir sozinha o que está acontecendo. Primeiro, o próprio código faz a análise com regras simples. Depois, o modelo Llama recebe esses dados e gera um relatório para a equipe de controle da missão.

In [37]:
def gerar_analise_ia(dados, alertas, decisoes, nivel_risco):
    modulos = dados["modulos"]

    prompt_usuario = f"""
Dados atuais da missão espacial experimental:

Temperatura dos módulos: {dados["temperatura"]}°C
Nível de energia: {dados["energia"]}%
Comunicação com a base: {dados["comunicacao"]}

Status dos módulos:
- Suporte à vida: {modulos["suporte_vida"]}
- Navegação: {modulos["navegacao"]}
- Propulsão: {modulos["propulsao"]}

Alertas identificados pelo sistema:
{alertas}

Decisões automáticas sugeridas pelo sistema:
{decisoes}

Nível de risco calculado: {nivel_risco}

Com base apenas nesses dados, escreva um relatório curto para a equipe de controle da missão.

O relatório deve:
1. resumir a situação atual;
2. apontar os principais riscos;
3. explicar as ações recomendadas;
4. finalizar com uma conclusão objetiva sobre o estado da missão.

Não invente informações que não estejam nos dados.
Não crie nome de missão, astronautas ou eventos externos.
"""

    resposta = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
                "content": """
Você é uma IA auxiliar de um sistema de controle de missão espacial experimental.

Sua função é transformar dados técnicos, alertas e decisões automáticas em um relatório claro para a equipe de controle.

Regras:
- responda sempre em português;
- use somente os dados fornecidos;
- não invente informações;
- seja objetivo;
- mantenha um tom técnico, mas fácil de entender;
- não exagere os riscos além do que foi informado.
"""
            },
            {
                "role": "user",
                "content": prompt_usuario
            }
        ]
    )

    return resposta["message"]["content"]

In [38]:
def exibir_relatorio(dados, alertas, decisoes, nivel_risco, analise_ia):
    modulos = dados["modulos"]

    print("=" * 60)
    print("MISSION CONTROL AI — STATUS DA MISSÃO")
    print("=" * 60)

    print("\nDADOS OPERACIONAIS")
    print("-" * 60)
    print(f"Temperatura dos módulos : {dados['temperatura']}°C")
    print(f"Nível de energia        : {dados['energia']}%")
    print(f"Comunicação             : {dados['comunicacao']}")
    print(f"Suporte à vida          : {modulos['suporte_vida']}")
    print(f"Navegação               : {modulos['navegacao']}")
    print(f"Propulsão               : {modulos['propulsao']}")

    print("\nALERTAS DO SISTEMA")
    print("-" * 60)
    for alerta in alertas:
        print(f"- {alerta}")

    print("\nDECISÕES AUTOMÁTICAS")
    print("-" * 60)
    for decisao in decisoes:
        print(f"- {decisao}")

    print("\nNÍVEL DE RISCO")
    print("-" * 60)
    print(nivel_risco)

    print("\nANÁLISE GERADA PELA IA")
    print("-" * 60)
    print(analise_ia)

    print("=" * 60)

In [39]:
dados = gerar_dados_missao()

alertas, decisoes, nivel_risco = analisar_missao(dados)

analise_ia = gerar_analise_ia(dados, alertas, decisoes, nivel_risco)

exibir_relatorio(dados, alertas, decisoes, nivel_risco, analise_ia)

MISSION CONTROL AI — STATUS DA MISSÃO

DADOS OPERACIONAIS
------------------------------------------------------------
Temperatura dos módulos : 103°C
Nível de energia        : 14%
Comunicação             : perdida
Suporte à vida          : normal
Navegação               : atenção
Propulsão               : falha

ALERTAS DO SISTEMA
------------------------------------------------------------
- Temperatura crítica detectada.
- Nível de energia crítico.
- Comunicação perdida com a base.
- Módulo de navegação requer atenção.
- Falha detectada no módulo de propulsão.

DECISÕES AUTOMÁTICAS
------------------------------------------------------------
- Ativar sistema de resfriamento de emergência.
- Ativar modo de economia de energia.
- Ativar protocolo de reconexão automática.
- Monitorar o módulo de navegação continuamente.
- Isolar o módulo de propulsão e iniciar diagnóstico técnico.

NÍVEL DE RISCO
------------------------------------------------------------
ALTO

ANÁLISE GERADA PELA IA


## Cenário 1 — Situação crítica simulada

Neste teste, os dados foram definidos manualmente para simular uma situação de risco.

A missão apresenta temperatura elevada, energia baixa, comunicação instável e falha no módulo de propulsão. Esse cenário demonstra a geração de alertas, decisões automáticas e análise da IA.

In [40]:
situacao_de_emergencia = {
    "temperatura": 95,
    "energia": 18,
    "comunicacao": "instável",

    "modulos": {
        "suporte_vida": "atenção",
        "navegacao": "normal",
        "propulsao": "falha"
    }
}

alertas, decisoes, nivel_risco = analisar_missao(situacao_de_emergencia)

analise_ia = gerar_analise_ia(situacao_de_emergencia, alertas, decisoes, nivel_risco)

exibir_relatorio(situacao_de_emergencia, alertas, decisoes, nivel_risco, analise_ia)

MISSION CONTROL AI — STATUS DA MISSÃO

DADOS OPERACIONAIS
------------------------------------------------------------
Temperatura dos módulos : 95°C
Nível de energia        : 18%
Comunicação             : instável
Suporte à vida          : atenção
Navegação               : normal
Propulsão               : falha

ALERTAS DO SISTEMA
------------------------------------------------------------
- Temperatura crítica detectada.
- Nível de energia crítico.
- Comunicação instável.
- Módulo de suporte à vida requer atenção.
- Falha detectada no módulo de propulsão.

DECISÕES AUTOMÁTICAS
------------------------------------------------------------
- Ativar sistema de resfriamento de emergência.
- Ativar modo de economia de energia.
- Redirecionar sinal para antena reserva.
- Monitorar o módulo de suporte à vida continuamente.
- Isolar o módulo de propulsão e iniciar diagnóstico técnico.

NÍVEL DE RISCO
------------------------------------------------------------
ALTO

ANÁLISE GERADA PELA IA
--

In [41]:
print("EXECUTANDO 3 SIMULAÇÕES AUTOMÁTICAS DA MISSÃO\n")

for i in range(1, 4):
    print(f"\nSIMULAÇÃO {i}")
    print("-" * 60)

    dados = gerar_dados_missao()
    alertas, decisoes, nivel_risco = analisar_missao(dados)
    analise_ia = gerar_analise_ia(dados, alertas, decisoes, nivel_risco)

    exibir_relatorio(dados, alertas, decisoes, nivel_risco, analise_ia)

    time.sleep(2)

EXECUTANDO 3 SIMULAÇÕES AUTOMÁTICAS DA MISSÃO


SIMULAÇÃO 1
------------------------------------------------------------
MISSION CONTROL AI — STATUS DA MISSÃO

DADOS OPERACIONAIS
------------------------------------------------------------
Temperatura dos módulos : 17°C
Nível de energia        : 84%
Comunicação             : instável
Suporte à vida          : normal
Navegação               : normal
Propulsão               : normal

ALERTAS DO SISTEMA
------------------------------------------------------------
- Comunicação instável.

DECISÕES AUTOMÁTICAS
------------------------------------------------------------
- Redirecionar sinal para antena reserva.

NÍVEL DE RISCO
------------------------------------------------------------
MÉDIO

ANÁLISE GERADA PELA IA
------------------------------------------------------------
**Relatório para a Equipe de Controle**

**Situção Atual:**
A missão experimental está operando normalmente com todos os módulos em funcionamento. A temperatura dos mó

## Cenário 2 — Missão estável

Neste teste, os dados representam uma missão em condição normal.

A temperatura, a energia, a comunicação e os módulos estão dentro do esperado. Esse cenário mostra que o sistema também consegue reconhecer uma operação sem alertas críticos.

In [42]:
dados_estaveis = {
    "temperatura": 45,
    "energia": 82,
    "comunicacao": "estável",

    "modulos": {
        "suporte_vida": "normal",
        "navegacao": "normal",
        "propulsao": "normal"
    }
}

alertas, decisoes, nivel_risco = analisar_missao(dados_estaveis)

analise_ia = gerar_analise_ia(dados_estaveis, alertas, decisoes, nivel_risco)

exibir_relatorio(dados_estaveis, alertas, decisoes, nivel_risco, analise_ia)

MISSION CONTROL AI — STATUS DA MISSÃO

DADOS OPERACIONAIS
------------------------------------------------------------
Temperatura dos módulos : 45°C
Nível de energia        : 82%
Comunicação             : estável
Suporte à vida          : normal
Navegação               : normal
Propulsão               : normal

ALERTAS DO SISTEMA
------------------------------------------------------------
- Nenhum alerta crítico detectado.

DECISÕES AUTOMÁTICAS
------------------------------------------------------------
- Manter operação padrão da missão.

NÍVEL DE RISCO
------------------------------------------------------------
BAIXO

ANÁLISE GERADA PELA IA
------------------------------------------------------------
**Relatório para a Equipe de Controle**

**Situação Atual:**
A missão espacial experimental está operando normalmente, com todos os módulos em funcionamento adequados. A temperatura dos módulos mantém-se dentro do limite recomendado (45°C), o nível de energia é estável e a comunicaçã

## Conclusão da prova de conceito

O Mission Control AI é uma prova de conceito para o monitoramento básico de uma missão espacial experimental.

O sistema gera dados simulados, analisa parâmetros como temperatura, energia, comunicação e status dos módulos, identifica situações críticas e sugere decisões automáticas com base em regras simples.

A IA generativa foi usada para transformar os dados técnicos e alertas em uma análise textual mais clara para a equipe de controle da missão. Dessa forma, a decisão principal é calculada pelo próprio código, enquanto a IA atua como apoio na interpretação dos resultados.

Com isso, o projeto atende aos requisitos principais da Sprint: uso de IA integrada, monitoramento de múltiplos parâmetros, geração de alertas, tomada de decisão básica e apresentação organizada dos resultados no Google Colab.